
# Database normalization with SMT solvers

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve
certain problems. Particularly, we will show how we can encode functional dependencies
in Z3 to determine if a relation is normalized.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver
from math import log
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Boolean

Suppose you have the following three boolean constraints and you want to check if there's a solution (an assignment of the variables) that satisfies all of them:

$$ x_1 \lor x_2 \lor x_3 $$

$$ \neg x_1 \implies \neg x_2$$

$$  x_1 \land x_3  $$

Let's see how we can do this using Z3.

In [ ]:
s = Solver() # initialize Z3 solver

# initialize variables

x1 = Bool('x_1') # declaring that x_1 is a boolean variable in Z3 which will be referred to as x1 in Python
x2 = Bool('x_2')
x3 = Bool('x_3')

# Note: we can also initialize multiple variables like so: x1, x2, x3 = Bools('x_1 x_2 x_3')

# we use s.add(.) to add a constraint to our solver s
# constraints can be made using many different operations such as Or, And, Not,
# equality, etc.

# here's how we would add the constraints above to our solver:

s.add( Or( x1, x2, x3 ) ) # add the first constraint
s.add( Implies( Not(x1), Not(x2) ) ) # add the second constraint
s.add( And( x1, x3 ) ) # add the third constraint

In [ ]:
# to view the constraints in our solver, we can use the following:
print( s )
# this prints the constraints as they appear in Z3 using Z3's notation

For better readability, this notebook also has a custom print function to view our constraints in LaTeX format, like so:

In [ ]:
showSolver( s )

In [ ]:
# we can use s.check() to run the solver and check whether its satisfiable:
print ( s.check() )

 "sat" means our system of constraints is satisfiable

In [ ]:
# after using s.check(),  we can use s.model() to output a solution if one exsits
solution = s.model()
print( solution )

Let's modify our system of constraint a bit and see if it's still satisfiable. Suppose we want to check if there's a solution where $x_1 = \neg x_3$. Let's see how we would do this with Z3.

In [ ]:
s.add( x1 == Not(x3) )
showSolver( s )

In [ ]:
print( s.check() ) # check if solution exists with new constraint

"unsat" means the system is not satisfiable, i.e., there is no assignment on the variables $x_1$, $x_2$, and $x_3$ that satisfies all the constraints we gave to the solver. **Note that if we were to run s.model() now we would get an error.**


## Database Normalization in Z3

In order to determine whether or not a relation is normalized in second normal form, we must first know the functional dependencies.
Below is an example relation with 4 attributes:

$$ R_1(\underline{studentID}, \underline{courseID}, studentName, courseFee) $$

The relation has the following functional dependencies:

$$ studentID \rightarrow studentName $$
$$ courseID \rightarrow courseFee $$

It is clear that this relation is not normalized, because studentName and courseFee have partial dependcies on the primary key. We will use Z3 to prove this


### Encoding Functional Dependencies

For two attributes X and Y, we say that X determines Y, $ X \rightarrow Y $, if for any two rows in the relation:

$$ row1.X = row2.X \rightarrow row1.Y = row2.Y $$

In other words, each value of X uniquely determines a value for Y.

In order to represent functional dependencies in Z3, we first need to create variables to represent two rows in the relation.
In the code below, we have created one variable for each attribute, for two arbitrary rows 'A' and 'B'.

No code is needed for this cell, **simply run the cell to create the variables**.


In [ ]:

# Create variables to represent two arbitrary rows in the relation
row1 = {
    'studentID' : String('A.studentID'),
    'courseID' : String('A.courseID'),
    'studentName' : String('A.studentName'),
    'courseFee' : String('A.courseFee'),
}
row2 = {
    'studentID' : String('B.studentID'),
    'courseID' : String('B.courseID'),
    'studentName' : String('B.studentName'),
    'courseFee' : String('B.courseFee'),
}



Next, we will write a function to represent a functional dependency in the relation.
We will do this using the definition of a functional dependency.

**Complete the code below** to create a function that encodes a functional dependendency in Z3.


In [ ]:

def functional_dependency(X, Y):
    return Implies( False ) # REPLACE THIS LINE    

# Initialize solver
s = Solver()

# Add two functional dependencies
s.add( functional_dependency('studentID', 'studentName') )
s.add( functional_dependency('courseID', 'courseFee') )

showSolver(s)



Great! Next, to determine if a relation is normalized, we must check whether or not certain dependencies exist.

We can do this by adding the following constraint:

$$ row1.X = row2.X \land row1.Y \neq row2.Y $$

Here, you can see we are asking the solver to find a case where the two rows share the same value of X, but they do not have the same value for Y. 
This would mean there is a counterexample where X does not determine Y. If no such counterexample exists, then we have proven the functional dependency exists.

**Complete the code below** to create a function that tests whether or not a functional dependency exists.


In [ ]:

def has_fd(s, X, Y):
    s.push()

    s.add(
        And( False ) # REPLACE THIS LINE
    )

    result = s.check() == unsat # If we can't find a counterexample, then X -> Y

    # Remove the most recent constraint
    s.pop()

    return result

# Initialize solver
s = Solver()

# Add a functional dependency
s.add( functional_dependency('studentID', 'studentName') )

# Check if the FD exists:
print("studentID -> studentName :", has_fd(s, 'studentID', 'studentName'))

# Check for a non-existent FD:
print("studentID -> courseFee :", has_fd(s, 'studentID', 'courseFee'))



### Congratulations! You have now learned how to use Z3 to encode functional dependencies!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm